In [ ]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Default paramaters
SEED = 460
S0 = 100
K = 95
R = 0.03
Q = 0.02
SIGMA = 0.25
M = 50
T = 0.5
NUM_PATH = 20_000

In [ ]:
# Question 1 functions


def GBM(
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Generate GBM with annualized paramaters
    """

    # Set seed for reproducibility
    rng = np.random.default_rng(seed=seed)
    dt = terminal_time / time_steps

    # Generate all brownian motion paths using standard normal distribution
    Z = rng.standard_normal((time_steps, num_path))

    # Compute log returns for each path and for each time step
    drift = (interest_rate - dividend_yield - 0.5 * volatility**2) * dt
    diffusion = volatility * np.sqrt(dt) * Z
    log_returns = drift + diffusion

    # Cumlate the log returns to get the log stock price
    log_prices = np.vstack([np.zeros(num_path), np.cumsum(log_returns, axis=0)])

    # Convert log prices to actual stock prices
    stock_prices = initial_stock_price * np.exp(log_prices)

    return stock_prices


def Asian_Put(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    """
    Generate prices for Asian put option where stock prices are modeled using GBM
    """
    # Compute the stock price paths using GBM
    stock_prices = GBM(
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )

    # Compute the payoff of the Asian put option for each path
    average_prices = np.mean(stock_prices, axis=0)
    payoffs = np.maximum(strike_price - average_prices, 0)

    # The price of the Asian put option is the discounted payoff
    put_prices = np.exp(-interest_rate * terminal_time) * payoffs

    return put_prices


def Monte_Carlo_Simulation(
    strike_price=K,
    initial_stock_price=S0,
    interest_rate=R,
    volatility=SIGMA,
    time_steps=M,
    terminal_time=T,
    dividend_yield=Q,
    num_path=NUM_PATH,
    seed=SEED,
):
    put_prices = Asian_Put(
        strike_price,
        initial_stock_price,
        interest_rate,
        volatility,
        time_steps,
        terminal_time,
        dividend_yield,
        num_path,
        seed,
    )
    return put_prices, np.mean(put_prices)

In [ ]:
# Question 1 plots
put_prices, average_put_price = Monte_Carlo_Simulation()
print(f"Average Asian Put Price: {average_put_price:.4f}")

time_grid = np.linspace(0, T, put_prices.shape[0])

plt.figure(figsize=(10, 6))
plt.plot(time_grid, paths[:, :50], alpha=0.3, linewidth=0.5)
plt.xlabel("Time (years)")
plt.ylabel("Stock Price (£)")
plt.title("Monte Carlo Stock Price Simulation - 50 of 10,000 Paths")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()